# weekday_baseline_features — patched v3 (fixes your KeyError)

Your current failure happens **after** `compute_level_weekly(...)`:

- Before: `sales_dense` columns are correct: `['date','store_id','product_id','category_name','qty']`
- After: `store_id` and `product_id` disappear from columns (they end up in an index level),
  and `sales_dense[["store_id", ...]]` raises `KeyError`.

**Root cause:** pandas `groupby().apply()` can return a MultiIndex and can silently move group keys
from columns into the index depending on internal execution paths / versions.

**Fix in v3:** we DO NOT use `groupby().apply()` at all.  
We compute rolling features with **vectorized** `groupby().rolling()` + `shift()` and explicitly
align back to the original row index. Keys remain normal columns.

No external files are written. All outputs are displayed in the notebook.


In [31]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd


## 1) Project root + CWD

In [32]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "raw_data").exists() and (p / "src").exists():
            return p
    raise RuntimeError(f"Project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print("CWD set to:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)


CWD set to: C:\Users\simon\food_prediction
PROJECT_ROOT: C:\Users\simon\food_prediction


## 2) Parameters

In [33]:
@dataclass(frozen=True)
class RunConfig:
    FILL_MISSING_QTY_WITH_ZERO: bool = True

    # Weekly level computation (leakage-safe)
    LOOKBACK_DAYS: int = 56
    LEVEL_METHOD: str = "mean"  # "median" for more robustness
    SHIFT_DAYS: int = 1
    WEEKLY_SUM_WINDOW: int = 7
    MIN_PERIODS_WEEKLY_SUM: int = 7
    MIN_PERIODS_LEVEL: int = 14


CFG = RunConfig()

RAW_DIR = PROJECT_ROOT / "raw_data"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("RAW_DIR:", RAW_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


RAW_DIR: C:\Users\simon\food_prediction\raw_data
REPORTS_DIR: C:\Users\simon\food_prediction\reports


## 3) Pick latest sales parquet

In [34]:
def pick_latest_file(folder: Path, pattern: str) -> Path:
    matches = sorted(folder.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files found: {folder} / {pattern}")
    return matches[-1]


SALES_FILE = pick_latest_file(RAW_DIR, "*_sales_data.parquet")
print("SALES_FILE:", SALES_FILE)


SALES_FILE: C:\Users\simon\food_prediction\raw_data\20260218_144523_sales_data.parquet


## 4) Load + standardize sales

We standardize to:
- `date` (normalized to midnight)
- `store_id`
- `product_id`
- `qty`
- `category_name`


In [35]:
def load_sales(sales_file: Path) -> pd.DataFrame:
    df = pd.read_parquet(sales_file).copy()

    if "date" not in df.columns:
        raise ValueError(f"Sales parquet has no 'date'. Available: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    rename_map = {}
    if "item_id" in df.columns and "product_id" not in df.columns:
        rename_map["item_id"] = "product_id"
    if "sold_quantity" in df.columns and "qty" not in df.columns:
        rename_map["sold_quantity"] = "qty"
    df = df.rename(columns=rename_map)

    required = {"date", "store_id", "product_id", "qty"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Sales missing required columns: {missing}. Available: {list(df.columns)}")

    if "category_name" not in df.columns:
        df["category_name"] = "UNKNOWN"

    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").fillna(0.0)

    return df


sales_raw = load_sales(SALES_FILE)
display(sales_raw.head(5))
display(sales_raw.describe(include='all'))
print("columns:", sales_raw.columns.tolist())


,date,category_name,product_id,qty,price,store_id
0,2025-04-01,Angebot Brötchen,139,15.0,0.0,0
1,2025-04-01,Angebot Feinbäckerei,138,28.0,0.0,0
2,2025-04-01,Angebot Heißgetränke,106,25.0,0.0,0
3,2025-04-01,Angebot Heißgetränke,539,5.0,1.4,0
4,2025-04-01,Angebot Snack,176,1.0,0.0,0


,date,category_name,product_id,qty,price,store_id
count,713637,713637,713637.000000,713637.000000,696648.000000,713637.000000
unique,NaN,23,NaN,NaN,NaN,NaN
top,NaN,Brötchen,NaN,NaN,NaN,NaN
freq,NaN,195190,NaN,NaN,NaN,NaN
mean,2025-05-15 16:04:01.050282240,NaN,262.957623,15.845915,2.747405,30.802189
min,2025-04-01 00:00:00,NaN,0.000000,-1.000000,-22.000000,0.000000
25%,2025-04-24 00:00:00,NaN,67.000000,1.000000,1.100000,14.000000
50%,2025-05-16 00:00:00,NaN,220.000000,4.000000,2.600000,30.000000
75%,2025-06-07 00:00:00,NaN,449.000000,12.000000,3.800000,46.000000
max,2025-06-30 00:00:00,NaN,676.000000,2425.000000,115.000000,83.000000


columns: ['date', 'category_name', 'product_id', 'qty', 'price', 'store_id']


## 5) Aggregate to unique daily keys per store×product

This guarantees uniqueness for `(date, store_id, product_id)`.


In [36]:
def stable_category(s: pd.Series) -> str:
    m = s.mode()
    if len(m):
        return str(m.iat[0])
    return str(s.iloc[0])


sales_daily = (
    sales_raw
    .groupby(["date", "store_id", "product_id"], as_index=False)
    .agg(
        qty=("qty", "sum"),
        category_name=("category_name", stable_category),
    )
)

assert not sales_daily.duplicated(["date", "store_id", "product_id"]).any(), "Duplicate daily keys still exist."

display(sales_daily.head(5))
print("Unique daily rows:", len(sales_daily))


,date,store_id,product_id,qty,category_name
0,2025-04-01,0,0,114.0,Brötchen
1,2025-04-01,0,1,0.0,Brötchen
2,2025-04-01,0,2,30.0,Brötchen
3,2025-04-01,0,5,0.0,Brötchen
4,2025-04-01,0,6,0.0,Brötchen


Unique daily rows: 711267


## 6) Dense daily grid (fill missing with 0)

Business rule:
- missing days mean `qty = 0` (“listed/open but nothing sold”).


In [37]:
def build_dense_daily_grid(sales_daily: pd.DataFrame, fill_zero: bool = True) -> pd.DataFrame:
    df = sales_daily.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    full_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")

    parts = []
    for (store_id, product_id), g in df.groupby(["store_id", "product_id"], sort=False):
        g = g.set_index("date").reindex(full_dates)
        g.index.name = "date"
        g = g.reset_index()

        g["store_id"] = store_id
        g["product_id"] = product_id

        if fill_zero:
            g["qty"] = g["qty"].fillna(0.0)

        g["category_name"] = g["category_name"].ffill().bfill().fillna("UNKNOWN")

        parts.append(g[["date", "store_id", "product_id", "category_name", "qty"]])

    dense = pd.concat(parts, ignore_index=True)
    dense = dense.reset_index(drop=True)  # hard guarantee
    return dense


sales_dense = build_dense_daily_grid(sales_daily, fill_zero=CFG.FILL_MISSING_QTY_WITH_ZERO)
display(sales_dense.head(10))
print("Dense rows:", len(sales_dense))
print("Dense columns:", sales_dense.columns.tolist())


,date,store_id,product_id,category_name,qty
0,2025-04-01,0,0,Brötchen,114.0
1,2025-04-02,0,0,Brötchen,149.0
2,2025-04-03,0,0,Brötchen,128.0
3,2025-04-04,0,0,Brötchen,144.0
4,2025-04-05,0,0,Brötchen,202.0
5,2025-04-06,0,0,Brötchen,0.0
6,2025-04-07,0,0,Brötchen,135.0
7,2025-04-08,0,0,Brötchen,146.0
8,2025-04-09,0,0,Brötchen,138.0
9,2025-04-10,0,0,Brötchen,155.0


Dense rows: 1713985
Dense columns: ['date', 'store_id', 'product_id', 'category_name', 'qty']


## 7) Leakage-safe weekly level (vectorized, no `apply()`)

We compute for each store×product:

1) `weekly_sum_t = rolling_sum_7d(qty)`  
2) `weekly_sum_leak_safe = weekly_sum_t.shift(SHIFT_DAYS)`  
3) `level_weekly = rolling_mean/median(weekly_sum_leak_safe, LOOKBACK_DAYS)`


In [38]:
def compute_level_weekly_vectorized(
    df: pd.DataFrame,
    lookback_days: int,
    level_method: str = "mean",
    shift_days: int = 1,
    weekly_sum_window: int = 7,
    min_periods_weekly_sum: int = 7,
    min_periods_level: int = 14,
) -> pd.DataFrame:
    if level_method not in {"mean", "median"}:
        raise ValueError("LEVEL_METHOD must be 'mean' or 'median'")

    needed = {"date", "store_id", "product_id", "qty"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"compute_level_weekly_vectorized requires {needed}, missing {missing}. Columns: {list(df.columns)}")

    out = df.sort_values(["store_id", "product_id", "date"]).copy()

    keys = ["store_id", "product_id"]
    g = out.groupby(keys, sort=False)

    # 1) Rolling weekly sum per group
    weekly_sum = g["qty"].rolling(window=weekly_sum_window, min_periods=min_periods_weekly_sum).sum()
    weekly_sum = weekly_sum.reset_index(level=keys, drop=True)

    # 2) Leakage-safe shift within group
    weekly_sum_leak_safe = weekly_sum.groupby([out["store_id"], out["product_id"]]).shift(shift_days)

    # 3) Rolling level on shifted weekly sum
    gb2 = weekly_sum_leak_safe.groupby([out["store_id"], out["product_id"]])
    if level_method == "mean":
        level = gb2.rolling(window=lookback_days, min_periods=min_periods_level).mean()
    else:
        level = gb2.rolling(window=lookback_days, min_periods=min_periods_level).median()
    level = level.reset_index(level=[0, 1], drop=True)

    out["level_weekly"] = level.astype(float)

    # Hard guarantee: keep keys as columns
    out = out.reset_index(drop=True)

    return out


sales_dense = compute_level_weekly_vectorized(
    sales_dense,
    lookback_days=CFG.LOOKBACK_DAYS,
    level_method=CFG.LEVEL_METHOD,
    shift_days=CFG.SHIFT_DAYS,
    weekly_sum_window=CFG.WEEKLY_SUM_WINDOW,
    min_periods_weekly_sum=CFG.MIN_PERIODS_WEEKLY_SUM,
    min_periods_level=CFG.MIN_PERIODS_LEVEL,
)

print("After level computation columns:", sales_dense.columns.tolist())
display(sales_dense[["date", "store_id", "product_id", "qty", "level_weekly"]].head(10))


After level computation columns: ['date', 'store_id', 'product_id', 'category_name', 'qty', 'level_weekly']


,date,store_id,product_id,qty,level_weekly
0,2025-04-01,0,0,114.0,NaN
1,2025-04-02,0,0,149.0,NaN
2,2025-04-03,0,0,128.0,NaN
3,2025-04-04,0,0,144.0,NaN
4,2025-04-05,0,0,202.0,NaN
5,2025-04-06,0,0,0.0,NaN
6,2025-04-07,0,0,135.0,NaN
7,2025-04-08,0,0,146.0,NaN
8,2025-04-09,0,0,138.0,NaN
9,2025-04-10,0,0,155.0,NaN


## 8) Notes on performance / RAM

`Dense rows` in your run are ~1.7M. Rolling computations create a few extra Series of the same length.

If this feels slow or RAM-heavy, a lighter alternative is to compute levels **without** densifying first
(i.e., only on observed sales days). But that changes your “missing day = 0” business rule.
